# Day 24: Cost & Token Optimization

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

Every token costs money on the cloud and latency on local. Most token waste in
agent systems is invisible: tool definitions resent on every call, conversation
history resent every turn, verbose prompts that license the model to over-explain.

Today is not about "write shorter prompts." That is a prompting skill. This
notebook demonstrates **five levers that measurably reduce tokens**, each with a
real before/after count read from Ollama's tokenizer, not a `len//4` guess.

## What You'll Build
- A real-token measurement helper (reads `prompt_eval_count` / `eval_count`)
- Five demonstrated optimization levers, each with measured savings
- A reusable playbook you can apply to any agent or agent workflow

## A note from Day 23

Day 23 showed LangGraph hands you real usage for free, the OpenAI Agents SDK
hides it (recoverable via a non-streamed replay), and CrewAI resisted the
documented recovery path. The cleanest real numbers on local Ollama come from
hitting the `/api/chat` endpoint directly, which is what the helper below does.
We use LangGraph to show one framework comparison, then switch to direct calls
for apples-to-apples lever measurements.

In [1]:
# ── Setup ──────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, "..")
from shared import (
    check_and_report, print_config, disable_openai_tracing,
    llm_chat_with_usage, get_langgraph_model,
)
check_and_report()
disable_openai_tracing()
print("=" * 50)
print("DAY 24 SETUP COMPLETE")
print("=" * 50)
print_config()

Python: 3.12.13



All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (10 model(s) available)

DAY 24 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


## First: stop estimating tokens

A common shortcut is `len(text) // 4` as a token estimate. For a notebook
about optimization that shortcut is self-defeating: you cannot optimize what
you mis-measure. The cell below shows the gap on one short prompt.

In [2]:
# len//4 vs the real tokenizer count from Ollama
sample = "What is the capital of Japan? One sentence."
estimate = len(sample) // 4
real = llm_chat_with_usage(
    [{"role": "user", "content": sample}], temperature=0.0
)
print(f"Prompt: {sample!r}")
print(f"len//4 estimate of input: {estimate}")
print(f"REAL input_tokens:        {real['input_tokens']}")
print(f"REAL output_tokens:       {real['output_tokens']}")
print(f"Estimate was off by:      {real['input_tokens'] / estimate:.1f}x")

# The helper reads prompt_eval_count (input) and eval_count (output) straight
# from Ollama's non-streamed response. Every number in this notebook is real.

Prompt: 'What is the capital of Japan? One sentence.'
len//4 estimate of input: 10
REAL input_tokens:        39
REAL output_tokens:       8
Estimate was off by:      3.9x


## The task and the ground rules

One task runs through every lever so the numbers are comparable: **"What is the
capital of Japan?"** against `qwen2.5:3b` via Ollama, `temperature=0.0` for
stable measurement. Output tokens vary slightly between runs even at temp 0;
the patterns below are stable. Your absolute numbers will land in the same range.

Each lever cell prints real `input / output / total` tokens before and after,
and computes the saving. Skip to the **Playbook** cell at the bottom for the
reusable checklist.

## Framework baseline (LangGraph, real `usage_metadata`)

LangGraph's `AIMessage.usage_metadata` is the one framework that gives real
token counts with no effort (Day 23 finding). The same verbose-vs-concise
comparison through LangGraph, reading the framework's own count:

In [3]:
from langchain_core.messages import SystemMessage, HumanMessage

model = get_langgraph_model()
QUERY = "What is the capital of Japan?"

verbose_sys_lg = (
    "You are a comprehensive geography expert. For each capital city question, "
    "provide the capital, country, population, continent, primary language, "
    "currency, founding year, major landmarks, climate, and cultural significance. "
    "Write at least 200 words."
)
concise_sys_lg = "Name the capital. One sentence max."

print(f"{'variant':<10} {'input':>6} {'output':>7} {'total':>7}  sample")
print("-" * 70)
for label, sysm in [("VERBOSE", verbose_sys_lg), ("CONCISE", concise_sys_lg)]:
    r = model.invoke([SystemMessage(content=sysm), HumanMessage(content=QUERY)])
    um = getattr(r, "usage_metadata", {}) or {}
    print(f"{label:<10} {um.get('input_tokens'):>6} {um.get('output_tokens'):>7} "
          f"{um.get('total_tokens'):>7}  {r.content[:50]!r}")

print()
print("On this single turn, concise beats verbose by ~10x on total tokens.")
print("OpenAI Agents SDK and CrewAI hide usage on local (see Day 23), so the")
print("lever cells below use direct Ollama calls for clean apples-to-apples counts.")

variant     input  output   total  sample
----------------------------------------------------------------------


VERBOSE        69     264     333  'The capital of Japan is Tokyo. \n\n- **Population:**'


CONCISE        28       8      36  'The capital of Japan is Tokyo.'

On this single turn, concise beats verbose by ~10x on total tokens.
OpenAI Agents SDK and CrewAI hide usage on local (see Day 23), so the
lever cells below use direct Ollama calls for clean apples-to-apples counts.


## Lever 1: Trim the system prompt

Strip filler ("you are a helpful assistant"), merge redundant constraints, drop
instructions the model already follows. This is the cheapest lever and the only
one the original version of this notebook demonstrated. Real numbers:

In [4]:
verbose_sys = (
    "You are a helpful geography assistant. When asked about capital cities, "
    "you should provide the capital, the country, the population of the city, "
    "the continent it's on, the language spoken, the currency used, and any "
    "interesting historical facts about the city. Be thorough and detailed."
)
concise_sys = "Answer geography questions. Give city, country, one fact. One sentence."

LEVER_RESULTS = []  # populated by each lever cell, printed by the summary cell

def show(label, messages, **kw):
    r = llm_chat_with_usage(messages, temperature=0.0, **kw)
    print(f"{label:<28} in={r['input_tokens']:>4}  out={r['output_tokens']:>4}  "
          f"total={r['total_tokens']:>4}  | {r['content'][:60]!r}")
    return r

print("LEVER 1: trim the system prompt")
print("-" * 70)
a = show("verbose system prompt",
         [{"role": "system", "content": verbose_sys}, {"role": "user", "content": QUERY}])
b = show("concise system prompt",
         [{"role": "system", "content": concise_sys}, {"role": "user", "content": QUERY}])
saving = (1 - b["total_tokens"] / a["total_tokens"]) * 100
print(f"\nSystem prompt savings: {a['total_tokens']} -> {b['total_tokens']} tokens "
      f"({saving:.0f}% total reduction). Output alone dropped {a['output_tokens']} -> "
      f"{b['output_tokens']} ({(1-b['output_tokens']/a['output_tokens'])*100:.0f}%).")
LEVER_RESULTS.append(("1. Trim system prompt", a["total_tokens"], b["total_tokens"],
                      "verbose -> concise prompt"))

LEVER 1: trim the system prompt
----------------------------------------------------------------------


verbose system prompt        in=  76  out= 388  total= 464  | 'The capital of Japan is Tokyo.\n\n### Population:\nTokyo has a '


concise system prompt        in=  35  out=  16  total=  51  | 'The capital of Japan is Tokyo, located in the Honshu island.'

System prompt savings: 464 -> 51 tokens (89% total reduction). Output alone dropped 388 -> 16 (96%).


## Lever 2: Constrain the output

Two ways to bound what comes back: a hard `max_tokens` ceiling, and a structured
output schema (JSON) that forces compact machine-readable output.

**The nuance:** these levers bite when the default is verbose. On an
already-brief answer they add overhead. The cell below runs both against the
*verbose* system prompt from Lever 1, which is where they earn their keep.

In [5]:
json_schema = {
    "type": "object",
    "properties": {
        "city": {"type": "string"},
        "country": {"type": "string"},
        "fact": {"type": "string"},
    },
    "required": ["city", "country", "fact"],
}
base = [{"role": "system", "content": verbose_sys}, {"role": "user", "content": QUERY}]

print("LEVER 2: constrain output (against the verbose system prompt)")
print("-" * 70)
a = show("verbose + free-form", base)
b = show("verbose + JSON schema", base, fmt=json_schema)
c = show("verbose + max_tokens=30", base, max_tokens=30)

def pct(x, y): return (1 - y / x) * 100
print(f"\nJSON schema:   output {a['output_tokens']} -> {b['output_tokens']} "
      f"({pct(a['output_tokens'], b['output_tokens']):.0f}% fewer output tokens).")
print(f"max_tokens=30: output {a['output_tokens']} -> {c['output_tokens']} "
      f"({pct(a['output_tokens'], c['output_tokens']):.0f}% fewer, but hard-truncated).")
print("\nTradeoff: max_tokens caps worst case but can cut mid-sentence.")
print("JSON keeps the answer usable and still removes most of the prose.")
LEVER_RESULTS.append(("2. Constrain output (JSON)", a["total_tokens"], b["total_tokens"],
                      "verbose free-form -> JSON schema"))
LEVER_RESULTS.append(("2. Constrain output (cap)", a["total_tokens"], c["total_tokens"],
                      "verbose -> max_tokens=30 (truncated)"))

LEVER 2: constrain output (against the verbose system prompt)
----------------------------------------------------------------------


verbose + free-form          in=  76  out= 388  total= 464  | 'The capital of Japan is Tokyo.\n\n### Population:\nTokyo has a '


verbose + JSON schema        in=  76  out=  74  total= 150  | '{ "city": "Tokyo", "country": "Japan", "fact": "The populati'


verbose + max_tokens=30      in=  76  out=  30  total= 106  | 'The capital of Japan is Tokyo.\n\n### Population:\nTokyo has a '

JSON schema:   output 388 -> 74 (81% fewer output tokens).
max_tokens=30: output 388 -> 30 (92% fewer, but hard-truncated).

Tradeoff: max_tokens caps worst case but can cut mid-sentence.
JSON keeps the answer usable and still removes most of the prose.


## Lever 3: Trim context per turn

In a multi-turn agent, the default is to resend the full conversation history
on every call. That input grows linearly with turns, and many models also
re-answer earlier turns. Trimming to a sliding window (or summarizing old turns)
is the biggest lever in long-running agents.

In [6]:
free_sys = "You are a geography assistant. Answer helpfully."
turns = [
    "What is the capital of France?",
    "What is its population?",
    "What language is spoken there?",
    "What currency is used?",
    "Name one famous landmark.",
    "What is the capital of Japan?",  # final, unrelated to history
]
full = [{"role": "system", "content": free_sys}] + [{"role": "user", "content": t} for t in turns]
trimmed = [{"role": "system", "content": free_sys}] + [{"role": "user", "content": t} for t in turns[-2:]]

print("LEVER 3: trim context (6-turn conversation)")
print("-" * 70)
a = show("full 6-turn history", full)
b = show("trimmed to last 2 turns", trimmed)
saving = (1 - b["total_tokens"] / a["total_tokens"]) * 100
print(f"\nTotal savings: {a['total_tokens']} -> {b['total_tokens']} tokens ({saving:.0f}%).")
print("In a real agent this grows with turn count: at 20 turns the gap is much wider.")
LEVER_RESULTS.append(("3. Trim context (6 turns)", a["total_tokens"], b["total_tokens"],
                      "full history -> last 2 turns"))

LEVER 3: trim context (6-turn conversation)
----------------------------------------------------------------------


full 6-turn history          in=  58  out= 176  total= 234  | 'The capital of France is Paris.\n- Population: As of 2021, Pa'


trimmed to last 2 turns      in=  35  out=  22  total=  57  | 'One famous landmark is the Eiffel Tower in Paris, France.\n\nT'

Total savings: 234 -> 57 tokens (76%).
In a real agent this grows with turn count: at 20 turns the gap is much wider.


## Lever 4: Right-size the model (routing)

Route cheap tasks (classification, routing, extraction) to a smaller model and
reserve the big one for generation. The price delta on cloud is often 5-10x.

**Capability tradeoff is real.** The smaller model is not a free win. The cell
below runs the same classification on `qwen2.5:3b` and `llama3.2:1b` and shows
both the token cost and the quality difference.

In [7]:
classify_task = (
    "Classify the intent of this query into one word: ROUTING, FAQ, or ESCALATION.\n\n"
    "Query: 'I want a refund for order 4521'"
)
print("LEVER 4: model routing on a classification task")
print("-" * 70)
a = show("qwen2.5:3b (bigger)", [{"role": "user", "content": classify_task}])
b = show("llama3.2:1b (smaller)", [{"role": "user", "content": classify_task}], model="llama3.2:1b")
saving = (1 - b["total_tokens"] / a["total_tokens"]) * 100
print(f"\nToken cost: {a['total_tokens']} -> {b['total_tokens']} ({saving:.0f}% fewer).")
print(f"3b output ({a['output_tokens']} tokens): {a['content'][:80]!r}")
print(f"1b output ({b['output_tokens']} tokens): {b['content'][:80]!r}")
print("Both models over-explained here; neither gave a clean one-word label. The")
print("smaller model is cheaper per token but is not automatically better at the task.")
print("Lesson: smaller is cheaper, but verify it can actually do the task.")
LEVER_RESULTS.append(("4. Smaller model (routing)", a["total_tokens"], b["total_tokens"],
                      "qwen2.5:3b -> llama3.2:1b (quality varies)"))

LEVER 4: model routing on a classification task
----------------------------------------------------------------------


qwen2.5:3b (bigger)          in=  65  out=  76  total= 141  | 'FAQLooking at the given query "I want a refund for order 452'


llama3.2:1b (smaller)        in=  59  out=  11  total=  70  | 'The intent of this query is "REFUND".'

Token cost: 141 -> 70 (50% fewer).
3b output (76 tokens): 'FAQLooking at the given query "I want a refund for order 4521", it appears to be'
1b output (11 tokens): 'The intent of this query is "REFUND".'
Both models over-explained here; neither gave a clean one-word label. The
smaller model is cheaper per token but is not automatically better at the task.
Lesson: smaller is cheaper, but verify it can actually do the task.


## Lever 5: Trim tool definitions

Tool schemas ride along in the system context on **every call**. Long
descriptions and unused parameters are pure input overhead, paid once per
request, every request. This is usually invisible because the cost is not in
the user's prompt.

In [8]:
verbose_tools = (
    "You have access to the following tools.\n\n"
    "Tool: get_weather\n"
    "Description: Returns the current weather for a given location. This includes "
    "temperature in Celsius, humidity percentage, wind speed, precipitation chance, "
    "and a short natural-language summary of conditions. Handles city names, "
    "postal codes, and airport codes. Falls back to nearest station if the exact "
    "location is unknown.\n"
    "Parameters:\n"
    "  - location (string, required): The full location name.\n"
    "  - units (string, optional): 'metric' or 'imperial'. Defaults to 'metric'.\n"
    "  - lang (string, optional): Language for the summary. Defaults to 'en'.\n\n"
    "Tool: calculator\n"
    "Description: Evaluates a mathematical expression and returns the numeric result. "
    "Supports addition, subtraction, multiplication, division, parentheses, and "
    "common functions like sqrt and pow. Follows standard operator precedence.\n"
    "Parameters:\n"
    "  - expression (string, required): The math expression to evaluate.\n"
)
minimal_tools = "Tools: get_weather(location)->str, calculator(expression)->str."
q = "What is 2+2?"

print("LEVER 5: trim tool definitions (unrelated task, tools still resent)")
print("-" * 70)
a = show("verbose tool defs",
         [{"role": "system", "content": verbose_tools}, {"role": "user", "content": q}])
b = show("minimal tool defs",
         [{"role": "system", "content": minimal_tools}, {"role": "user", "content": q}])
saving = (1 - b["total_tokens"] / a["total_tokens"]) * 100
print(f"\nInput savings: {a['input_tokens']} -> {b['input_tokens']} tokens ({saving:.0f}% fewer total).")
print("This overhead repeats on every single agent step. Multiply by step count.")
LEVER_RESULTS.append(("5. Trim tool definitions", a["total_tokens"], b["total_tokens"],
                      "verbose schemas -> one-line schemas"))

LEVER 5: trim tool definitions (unrelated task, tools still resent)
----------------------------------------------------------------------


verbose tool defs            in= 203  out=   9  total= 212  | '2+2 evaluates to 4.'


minimal tool defs            in=  33  out=   8  total=  41  | '2+2 equals 4.'

Input savings: 203 -> 33 tokens (81% fewer total).
This overhead repeats on every single agent step. Multiply by step count.


## On prompt caching (not demonstrated here)

Cloud providers (OpenAI, Anthropic) cache a static prefix (system prompt, tool
defs, few-shot examples) and serve cache hits at roughly 10x lower input cost.
On local Ollama the KV-cache helps latency but the API does not expose a cached
input-token rate, so a measured demo here would be theater.

In production: if your system prompt + tool defs are large and stable across
calls (most agents), prompt caching is the single biggest cloud cost lever after
context trimming. Mark cache breakpoints on the static prefix. We skip the demo
rather than fake a number.

## The token-saving playbook

Run this on any agent or agent workflow, in order. Each step has a measured cell
above. Re-measure after every change; the baseline moves.

1. **Measure real tokens first.** Not `len//4`. Read `usage_metadata`
   (LangGraph), `prompt_eval_count`/`eval_count` (Ollama), or a non-streamed
   replay (OpenAI Agents SDK, Day 23). No baseline, no optimization.
2. **Trim the system prompt.** Strip filler, merge constraints, drop unused
   instructions. Cheapest win. (Lever 1)
3. **Constrain the output.** `max_tokens` for a hard ceiling, structured output
   / JSON schema when the default is verbose. (Lever 2)
4. **Trim context per turn.** Sliding window, summarize old turns, or selective
   memory. Biggest lever in multi-turn agents. (Lever 3)
5. **Right-size the model.** Route classification / routing / extraction to a
   smaller model. Verify capability before you trust it. (Lever 4)
6. **Trim tool definitions.** Shorten descriptions, drop unused params. Paid on
   every call. (Lever 5)
7. **Cache the static prefix.** Cloud only; biggest lever there for agents with
   large stable system context. (See note above)
8. **Cut redundant calls.** Trace first (Day 23), then kill re-asks, re-fetches,
   and no-op loops.
9. **Re-measure.** Same task, before/after, real numbers. Report the delta.

The measured summary table from this notebook's runs is below.

In [9]:
# Consolidated measured results from this notebook's actual runs.
# Each lever cell above appended its before/after to LEVER_RESULTS.
print(f"{'Lever':<32} {'before':>7} {'after':>7} {'saving':>8}  note")
print("-" * 95)
for name, before, after, note in LEVER_RESULTS:
    pct_s = f"{(1-after/before)*100:.0f}%"
    print(f"{name:<32} {before:>7} {after:>7} {pct_s:>8}  {note}")
print()
print("Numbers above are from single runs at temperature=0 on qwen2.5:3b")
print("(Lever 4 'after' side: llama3.2:1b). Re-running this notebook produces")
print("numbers in the same range; the patterns are stable.")

Lever                             before   after   saving  note
-----------------------------------------------------------------------------------------------
1. Trim system prompt                464      51      89%  verbose -> concise prompt
2. Constrain output (JSON)           464     150      68%  verbose free-form -> JSON schema
2. Constrain output (cap)            464     106      77%  verbose -> max_tokens=30 (truncated)
3. Trim context (6 turns)            234      57      76%  full history -> last 2 turns
4. Smaller model (routing)           141      70      50%  qwen2.5:3b -> llama3.2:1b (quality varies)
5. Trim tool definitions             212      41      81%  verbose schemas -> one-line schemas

Numbers above are from single runs at temperature=0 on qwen2.5:3b
(Lever 4 'after' side: llama3.2:1b). Re-running this notebook produces
numbers in the same range; the patterns are stable.


## Key takeaways

- **Measure before you optimize.** `len//4` underestimated real input tokens by
  4x on a single short prompt. You cannot cut what you mis-count.
- **The biggest hidden cost is not the model, it is resent overhead**: tool
  definitions on every call (Lever 5), conversation history every turn (Lever 3).
  These do not show up in your prompt.
- **Output is usually more expensive than input** on cloud pricing. Bounding it
  with `max_tokens` or structured output is high-impact.
- **Smaller models are cheaper, not free.** Verify they can do the task before
  routing production traffic to them.
- On local Ollama the cost is latency, not dollars. The same levers apply:
  fewer tokens, faster response, better UX.

---

## LinkedIn Post Draft

I spent Day 24 measuring where tokens actually leak in agent systems.

Same task, same local model (qwen2.5:3b via Ollama), real token counts read
from the tokenizer. Not len-divided-by-4 guesses, which underestimated input
tokens by 4x on a one-line prompt.

Five levers, measured before and after:

Trim the system prompt: 464 to 51 tokens on one call. Strip the filler the
model already follows.

Constrain the output: against a verbose prompt, a JSON schema cut output from
388 to 74 tokens. A max_tokens cap hard-cuts output to 30 but can truncate
mid-sentence.

Trim context per turn: a 6-turn conversation cost 234 tokens when the full
history was resent, 57 when trimmed to the last two. This scales with turn
count, so it is the biggest lever in long-running agents.

Right-size the model: a classification task cost 141 tokens on qwen2.5:3b and
70 on llama3.2:1b. The smaller model was terser but still gave the wrong label.
Cheaper is not free.

Trim tool definitions: 212 tokens with verbose tool schemas resent on every
call, 41 with one-line schemas. This overhead is invisible because it is not in
your prompt, and it repeats on every agent step.

The biggest token waste in agents is not the model. It is the stuff you do not
see: tool definitions resent every call, and conversation history resent every
turn.

The playbook is the same every time. Measure real tokens. Cut the prompt. Cap
the output. Trim history. Route to a smaller model where it can do the job.
Shrink tool schemas. Re-measure.

What is leaking tokens in your agent that you haven't measured yet?

#AIAgents #TokenOptimization #LLMOps #30DayChallenge